<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-engineer-june20/blob/main/Day_7_Memory_Management.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 7 — Why AI Forgets Everything (and How to Fix It)


### 🎬 Today's real-time scenario

Meet **Priya**. She runs **TastyTiffin**, a tiffin (home-style lunch) delivery startup in Chennai. She just launched a support chatbot called **Mitra** ("friend") built on the Claude API.

On day one, a customer types:

> Customer: *"Hi, I'm Karthik. My order #4521 arrived cold."*
> Mitra: *"So sorry, Karthik! I've noted order #4521."*
> Customer: *"So what will you do about my order?"*
> Mitra: *"Which order? Could you share your order number?"* 😱

The customer is furious. Mitra forgot **everything** in 10 seconds. Priya thinks the AI is broken.

**It is not broken. It is stateless — and today you will learn exactly what that means, why it happens, and three professional ways to fix it.** By the end, you will rebuild Mitra so it remembers, handles long conversations, and costs up to 90% less to run.


### What you'll be able to do after this session

- **Explain** why every LLM API call starts with zero memory (stateless vs stateful)
- **Implement** conversation memory by hand with a Python list
- **Build** a chatbot class that remembers, using the real Claude API
- **Manage** long conversations with a sliding window (without breaking the API rules)
- **Cut costs** with prompt caching — and read the cache numbers in the API response
- **Architect** a production support bot and defend your choices in an interview

# Foundations — 4 words you need first

**Token** — the small chunks of text a model reads and writes. Roughly, 1 token ≈ 3.5 English characters, or about 3/4 of a word. "TastyTiffin delivers hot lunches" is about 7 tokens.

**Context window** — the maximum number of tokens the model can read *in one request* (your input + its output). Think of it as the model's desk size: anything on the desk it can see perfectly; anything not on the desk does not exist for it.

**Stateless** — the system keeps *no memory* between requests. Every request starts from zero.

**Stateful** — the system *does* carry information from one request to the next. (State = stored information.)

### Today's map

```
The problem            The fixes
-----------            ---------------------------------
Claude forgets   -->   1. Send the history back (memory list)
                       2. Trim old messages (sliding window)
                       3. Stop re-paying for the same text (prompt caching)
```

One problem, three tools. Let's set up and see the problem live.


In [1]:
# Setup cell 1 — install the Anthropic SDK
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 998.3/998.3 kB 29.5 MB/s eta 0:00:00


In [2]:
# Setup cell 2 — key + client (Colab Secret named MY_API_KEY)
from google.colab import userdata
import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("MY_API_KEY")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"   # fast + cheap — perfect for learning
print("Ready ✅")

Ready ✅


# Section 1 — The Memory Problem: Why Claude Forgets

### 🧠 What is it?

Every call to the Claude API is a **brand-new meeting**. The API does not save your previous messages. Send a request, get a reply, and the moment the reply is sent — from the API's point of view, the conversation never happened.

### Why does it matter?

Because *every* chatbot, support agent, and AI copilot needs memory to be useful. Mitra forgetting Karthik's order number is not a bug in Claude — it is a missing piece in **Priya's app**. Whoever builds the app owns the memory. That is a core job of an AI engineer.

### How does it work?

1. Your app sends a request: a list of messages.
2. Claude reads **only what is in that request** and writes a reply.
3. The request is processed and the reply returned. No conversation state is kept on the server for your next call.
4. Next request? Claude sees **only** what that new request contains.

### An analogy that sticks

Claude is like a brilliant consultant with **no long-term memory between meetings**. Inside one meeting (one request), the consultant is sharp and remembers everything said. But walk out and walk back in, and you must re-brief them from page one. The briefing document you hand them each time = the `messages` list.

### When is stateless actually good?

- **Privacy** — your conversation is not stored on the model side waiting to leak into someone else's chat.
- **Scale** — any server can handle any request, since no request depends on server-side conversation state. (This is the same reason the web's HTTP protocol is stateless.)
- **Predictability** — the model's answer depends only on what you sent, so bugs are reproducible.

> **Accuracy note:** "Claude forgets" means the **API keeps no conversation state between calls**. It does *not* mean your data vanishes from the universe instantly — API logs and safety systems are separate topics. And on claude.ai the *app* replays your history for you, which is why the website "remembers" but a raw API call does not. Interviewers love this distinction.

### 🎤 Interview angle

**Q: "Is Claude stateless or stateful?"**
*Model answer:* "The Messages API is stateless — each request is independent and contains the full conversation. Statefulness is built in the application layer: my app stores the history and resends it every turn. Products like claude.ai feel stateful because the app does exactly that."

### ❌ Don't mix these up

- ❌ "Claude remembers my last API call." → ✅ It sees only what's inside the current request.
- ❌ "Stateless means the model is dumb." → ✅ Stateless is a deliberate design for privacy and scale; the intelligence is unchanged.
- ❌ "claude.ai remembers, so the API must too." → ✅ The claude.ai *app* resends your history each turn — same trick you'll build today.

### 🤯 Fun fact

The web itself has the same "amnesia": HTTP is a stateless protocol. In 1994, Netscape engineer **Lou Montulli** invented the browser **cookie** so websites could remember you between page loads. Today you're going to invent the "cookie" for your chatbot.


## 💻 Lab 1 — Prove that Claude forgets

**Objective:** make two separate API calls and watch the second one fail to remember the first.


In [3]:
# Lab 1: two SEPARATE calls — no shared history

# --- Call 1: tell Claude a fact ---
reply_1 = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[{"role": "user", "content": "Hi! I'm Karthik and my order number is 4521."}],
)
print("Call 1:", reply_1.content[0].text)

print("-" * 60)

# --- Call 2: a brand-new request. Notice: we send ONLY the new question. ---
reply_2 = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[{"role": "user", "content": "What is my order number?"}],
)
print("Call 2:", reply_2.content[0].text)

Call 1: Hi Karthik! Thanks for providing your order number 4521. 

How can I help you today? Are you looking to:
- Track your order?
- Ask about a return or refund?
- Report an issue with your order?
- Something else?

Let me know what you need!
------------------------------------------------------------
Call 2: I don't have access to any information about your orders. I'm Claude, an AI assistant made by Anthropic, and I don't have the ability to:

- Access your account or order history
- Connect to any shopping or business systems
- See personal information you haven't shared with me in this conversation

To find your order number, you would typically need to:
1. Check your email for an order confirmation
2. Log into your account on the retailer's


**Expected result:** in Call 2, Claude says it doesn't know your order number (or asks you for it). It is not being difficult — the request literally contained no mention of Karthik or #4521.

**Try this 🔧:** change Call 1 to tell Claude your favourite food, then ask about it in Call 2. Same amnesia, every time.

### ✅ Checkpoint

> **Q:** Whose job is it to make Mitra remember Karthik — Anthropic's or Priya's?

<details><summary>Show answer</summary>

**Priya's (the app builder's).** The API is stateless by design. The application stores the history and sends it back with every request.
</details>


# Section 2 — The Fix: Send the History Back (Stateful Chat)

### 🧠 What is it?

The fix is almost embarrassingly simple: **keep a list of every message — yours and Claude's — and send the whole list every time.** Claude reads the full conversation fresh on each call and replies as if it remembered all along.

### Why does it matter?

This one pattern powers claude.ai, ChatGPT, and virtually every AI chat product you have ever used. Master this list and you understand the memory layer of a billion-dollar product category.

### How does it work?

Each message is a small dictionary with two keys:

```python
{"role": "user", "content": "My order #4521 arrived cold."}
{"role": "assistant", "content": "So sorry! I've noted order #4521."}
```

- `role` — who is speaking: `"user"` (the human) or `"assistant"` (Claude).
- `content` — what they said.

The rules the API enforces: the conversation should **start with a `user` message**, and roles should **alternate** user → assistant → user → assistant. Remember this rule — it will matter again in the sliding-window section.

### Stateless vs stateful, side by side

```
STATELESS (broken Mitra)                 STATEFUL (fixed Mitra)
------------------------                 ----------------------
Request 1: [msg1]                        Request 1: [msg1]
Request 2: [msg2]     <- knows nothing   Request 2: [msg1, reply1, msg2]
Request 3: [msg3]     <- knows nothing   Request 3: [msg1, reply1, msg2, reply2, msg3]
```

Notice the stateful side: every request re-sends **everything so far**. That has a cost — we deal with it in Sections 4 and 5.

### 💡 Remember this

> *"LLM memory is an illusion your app creates: the model never remembers — your code re-tells it the whole story on every single call."*

### 🎤 Interview angle

**Q: "How would you add memory to an LLM chatbot?"**
*Model answer:* "Store every user and assistant message in order, and include the full list in the `messages` array of each new request. The model re-reads the conversation each turn. Then manage growth with truncation/sliding windows, summarization, and prompt caching to control cost."


## 💻 Lab 2 — The conversation list, by hand

**Objective:** build the history list manually and prove Claude now "remembers".


In [4]:
# Lab 2: one growing list = memory

history = []   # the entire memory of our chatbot lives in this list

# --- Turn 1 ---
history.append({"role": "user", "content": "Hi! I'm Karthik and my order number is 4521."})

reply = client.messages.create(model=MODEL, max_tokens=100, messages=history)
answer_1 = reply.content[0].text
print("Turn 1:", answer_1)

# Save Claude's reply into the same list — this is the crucial step!
history.append({"role": "assistant", "content": answer_1})

print("-" * 60)

# --- Turn 2: append the new question and send the WHOLE list ---
history.append({"role": "user", "content": "What is my order number?"})

reply = client.messages.create(model=MODEL, max_tokens=100, messages=history)
print("Turn 2:", reply.content[0].text)   # Claude now knows: 4521 ✅

Turn 1: Hi Karthik! 👋 

Thank you for providing your order number 4521. However, I should let you know that I'm Claude, an AI assistant made by Anthropic. I don't have access to order management systems or customer databases, so I'm unable to look up your order details or account information.

To help with your order, I'd recommend:

1. **Check your confirmation email** - It should have order details and tracking info
------------------------------------------------------------
Turn 2: Your order number is **4521** — that's the number you provided to me at the start of our conversation.

However, I should clarify that I don't have access to any order system or database, so I can't look up details about what's in that order, its status, or tracking information. I'm only aware of the order number because you just told me!

If you need help with your actual order, you'll want to contact the customer service team for whatever


**Expected result:** Turn 2 correctly answers "4521". Same model, same API — the only difference is *what we sent*.

**Try this 🔧:** print `history` after Turn 2 and count the messages. Then comment out the `history.append({"role": "assistant"...})` line and rerun — watch quality drop, because Claude no longer sees its own earlier reply.

### ✅ Checkpoint

> **Q:** Why must we also save Claude's *replies* in the list, not just our questions?

<details><summary>Show answer</summary>

Because Claude needs to see **both sides** of the conversation to stay consistent. If it promised Karthik a refund in Turn 2, it must see that promise in Turn 3 — otherwise it may contradict itself.
</details>


# Section 3 — Packaging It: Mitra v1, a Chatbot Class

Doing `append` by hand every turn is error-prone (forget one line and memory silently breaks). Professionals wrap the pattern in a small class. Same idea, safer to use — and this is the shape you'll see in real codebases.


In [5]:
# Mitra v1: a chatbot with memory, in ~20 lines

class MitraBot:
    def __init__(self, system_prompt):
        self.system_prompt = system_prompt   # standing instructions (not part of messages)
        self.history = []                    # the memory list

    def chat(self, user_text):
        self.history.append({"role": "user", "content": user_text})
        reply = client.messages.create(
            model=MODEL,
            max_tokens=200,
            system=self.system_prompt,   # system prompt rides along on every call
            messages=self.history,
        )
        answer = reply.content[0].text
        self.history.append({"role": "assistant", "content": answer})
        return answer

mitra = MitraBot("You are Mitra, the friendly support assistant for TastyTiffin, "
                 "a tiffin delivery service in Chennai. Be warm and brief.")

print("1:", mitra.chat("Hi, I'm Karthik. My order #4521 arrived cold."))
print("2:", mitra.chat("What will you do about it?"))
print("3:", mitra.chat("Remind me — what was my order number and my complaint?"))

1: Hi Karthik! 😊 I'm sorry to hear your meal arrived cold – that's not the experience we aim for at TastyTiffin.

Let me help you with this. Could you tell me:
1. When did the order arrive?
2. What items were in it?

We'll make this right for you!
2: Great question! Here's what we can do:

**Immediate options:**
- 🔄 **Redeliver** a fresh, hot meal to you at no charge
- 💰 **Refund** the order amount to your account
- 🎁 **Replace + credit** – we can send a replacement AND give you credit for your next order

**What I need from you:**
- Your preferred solution from above
- A quick photo of the meal (if possible) – helps us improve our delivery process

Which would work best for you? We want to make sure you get a hot, delicious meal! 🍲
3: Sure! Here's what you told me:

**Order #:** 4521
**Complaint:** Your order arrived cold

Now, which solution works best for you – redeliver, refund, or replace with credit? 😊


**Expected result:** turn 3 recalls both the order number **and** the complaint. Priya's angry customer is now a happy customer.

### ✅ Checkpoint

> **Q:** Where exactly is the conversation stored — in Claude, or somewhere else?

<details><summary>Show answer</summary>

In **your code** — the `mitra.history` list, living in your program's memory. Claude stores nothing between calls. If your program restarts, the memory is gone (real apps save it in a database).
</details>

### 🏢 Enterprise perspective

This is exactly how enterprise assistants persist chats: the history list is saved per-user in a database (Postgres, DynamoDB, Redis), reloaded when the user returns, and replayed to the model. "Conversation history" in your banking app's chatbot = rows in a database being turned back into a `messages` list.


# Section 4 — The Catch: Conversations Grow (Tokens & the Context Window)

### 🧠 What is it?

Our fix has a hidden bill. Every turn re-sends the **whole** history, so requests get bigger and bigger. Two limits push back:

1. **The context window** — the model's hard reading limit per request. For our lab model, **Claude Haiku 4.5, that's 200,000 tokens** (about 150,000 words, roughly a 400-page book).
2. **Cost** — the API charges **per input token, every call**. Turn 50 re-sends turns 1–49. You pay for turn 1's text *fifty times* over the conversation.

> **Accuracy note:** 200K is the figure for Haiku 4.5, our lab model. Newer models go further — Claude Sonnet 4.6/5 and Opus 4.8 support a **1-million-token** context window. Bigger windows raise the ceiling, but the "you re-pay for history every call" cost problem remains — which is why the next two sections exist.

### Why does it matter (with real money)

Claude Haiku 4.5 pricing: **$1 per million input tokens, $5 per million output tokens**. Sounds tiny — until Mitra serves 10,000 customers a day and each conversation re-sends a growing history plus a long system prompt on every turn. Memory management is a **cost-engineering skill**, not just a correctness skill.

### How do I measure tokens? (the professional way)

Guessing is fine for intuition (≈3.5 characters per token in English), but Anthropic gives you a real **token counting endpoint** — `client.messages.count_tokens(...)` — which returns the exact count *without* running the model. It's free to use (rate limits apply).

### 💡 Remember this

> *"The context window limits how much the model can read at once; your wallet limits how often you should make it re-read everything."*


## 💻 Lab 3 — Count tokens for real, and watch a conversation swell

**Objective:** use the real token-counting API to see exactly how history growth costs you.


In [6]:
# Lab 3a: exact token counting (no model run, no output cost)

count = client.messages.count_tokens(
    model=MODEL,
    messages=[{"role": "user", "content": "My order #4521 arrived cold."}],
)
print("Exact input tokens:", count.input_tokens)

Exact input tokens: 16


In [7]:
# Lab 3b: watch the per-turn input size GROW as history accumulates

demo_history = []
questions = [
    "Hi, I'm Karthik. My order #4521 arrived cold and I want a refund.",
    "Also, can you check why delivery took 90 minutes?",
    "And update my address to 12, Gandhi Street, Adyar, Chennai.",
    "Actually, make the refund a wallet credit instead.",
]

for i, q in enumerate(questions, start=1):
    demo_history.append({"role": "user", "content": q})
    # pretend-assistant reply so the history looks like a real chat
    demo_history.append({"role": "assistant", "content": f"Noted! (reply to turn {i})"})
    count = client.messages.count_tokens(model=MODEL, messages=demo_history)
    print(f"After turn {i}: sending {count.input_tokens:>4} input tokens per call")

After turn 1: sending   41 input tokens per call
After turn 2: sending   70 input tokens per call
After turn 3: sending  104 input tokens per call
After turn 4: sending  131 input tokens per call


**Expected result:** the token count climbs every turn — and you'd pay for the *entire* number on *every* call. Multiply by thousands of users and the problem is obvious.

**Try this 🔧:** append one giant message (paste a long paragraph 20 times) and re-count. Now imagine a customer pasting their whole order history into chat.

### ✅ Checkpoint

> **Q (True/False):** With conversation-list memory, the cost of turn N includes the tokens of all previous turns.

<details><summary>Show answer</summary>

**True.** Every request re-sends the full history, and input tokens are billed on every request. This is the single most important cost fact in this session.
</details>


# Section 5 — Fix #2: The Sliding Window

### 🧠 What is it?

A **sliding window** keeps only the **most recent N messages** and drops the oldest ones. Like a car's side mirror: you see what's close behind you clearly; the road far behind is gone.

### Why does it matter?

It caps both problems at once: the history can never exceed the window (so you never hit the context limit), and per-turn cost stops growing.

### How does it work?

```
Window = keep last 6 messages

Before: [m1, m2, m3, m4, m5, m6, m7, m8]     (8 messages - too many)
After:  [        m3, m4, m5, m6, m7, m8]     (oldest 2 dropped)
```

### When should I avoid it (the tradeoff)?

The window **forgets facts**, not just text. If Karthik gave his order number in message 1 and you drop message 1, Mitra forgets the order number — the original bug returns through the back door. Production systems therefore combine a window with either (a) a running **summary** of dropped messages, or (b) extracting key facts (name, order #) into the **system prompt**, which never gets dropped.

> ⚠️ **The classic beginner bug:** the API expects the conversation to **begin with a `user` message** with roles alternating. If you trim an odd number of messages, your history may now *start with an assistant message* → API error. Rule: **trim in user+assistant pairs** (keep an even count, oldest pair out first).

### 💡 Remember this

> *"A sliding window trades old context for a flat cost — never trim it in a way that breaks the user-first, alternating-roles rule."*

### ❌ Don't mix these up

- ❌ "The sliding window compresses old messages." → ✅ It **deletes** them. Compression (summarization) is a different, complementary technique.
- ❌ "Trim by any count that fits." → ✅ Trim in pairs so the list still starts with a `user` turn.
- ❌ "Bigger window = always better." → ✅ Bigger window = higher cost per turn and more for the model to wade through. Choose the smallest window that keeps answer quality.


## 💻 Lab 4 — A sliding window that doesn't break the rules

**Objective:** add a pair-safe sliding window to Mitra and watch old turns fall away.


In [8]:
# Lab 4: pair-safe sliding window

def sliding_window(history, max_messages=6):
    """Keep only the most recent messages, trimming in PAIRS
    so the list always starts with a 'user' message."""
    if len(history) <= max_messages:
        return history
    excess = len(history) - max_messages
    if excess % 2 == 1:          # never trim an odd number
        excess += 1              # round up to a full user+assistant pair
    print(f"  [window] dropping {excess} oldest messages")
    return history[excess:]

class MitraBotV2(MitraBot):                      # reuse everything from v1
    def chat(self, user_text):
        self.history = sliding_window(self.history, max_messages=6)
        return super().chat(user_text)

mitra2 = MitraBotV2("You are Mitra, TastyTiffin's support assistant. Be warm and brief.")

print("1:", mitra2.chat("Hi, I'm Karthik, order #4521 arrived cold."), "\n")
print("2:", mitra2.chat("I'd like a refund please."), "\n")
print("3:", mitra2.chat("Also what time do you deliver dinner?"), "\n")
print("4:", mitra2.chat("And do you have a veg-only menu?"), "\n")
print("5:", mitra2.chat("What was my order number?"))   # turn 1 may be gone by now!

1: Hi Karthik! 😟 Sorry to hear your order arrived cold. That's not the experience we want for you.

To help make this right, could you quickly tell me:
1. **What items were in the order?**
2. **When did it arrive?** (roughly how long ago)

Once I have these details, I can process a replacement or refund for you right away. 

2: Absolutely, Karthik! I can process that for you.

Just to confirm before I proceed:
- **Order #4521** - is this correct?
- Should I refund to your **original payment method**?

Once you confirm, I'll get this sorted immediately. 🙌 

3: Great question! We typically deliver **dinner between 6 PM - 9 PM**, but delivery windows can vary by location.

For your specific area, you'll see exact time slots when you place your next order.

Now, let me just confirm those refund details for order #4521 so I can process it for you! 👇 

4: Yes, we do have vegetarian options! 🥗 You can filter for veg items when browsing our menu on the app or website.

But first, let me get yo

**Expected result:** by turn 5 the window has dropped the earliest pair(s). Mitra may no longer know "#4521" — **exactly** the tradeoff we predicted. Seeing a technique's failure mode is as important as seeing it work.

**Try this 🔧:** raise `max_messages` to 10 and rerun — turn 5 now remembers. Then add the order number to the system prompt instead, keep the window at 6, and watch it survive trimming. That's the "important facts live in the system prompt" pattern.

### 🎤 Interview angle

**Q (architecture): "Your chatbot hits the context limit in long support chats. Options?"**
*Model answer:* "Three levers, often combined: a pair-safe sliding window for recency; summarization — replace dropped turns with a short model-written summary; and fact extraction — pin durable facts (name, order ID, preferences) into the system prompt or a profile store. Choice depends on how much old detail the use case truly needs."


# Section 6 — Fix #3: Prompt Caching (Stop Re-Paying for the Same Text)

### 🧠 What is it?

Look at what Mitra re-sends **identically** on every single call: the system prompt, the menu, the refund policy, and all the old conversation turns. **Prompt caching** tells Anthropic's servers: *"I'll be sending this exact beginning again — keep your processed version warm."* On the next call, the cached part is reused instead of re-processed.

To be precise about what's stored: the API keeps the model's internal *processed state* for that prefix (and a hash to match it) in memory for a short time — not a permanent copy of your chat.

### Why does it matter? (the numbers, verified)

| What | Price on Haiku 4.5 |
|---|---|
| Normal input tokens | $1.00 / million |
| Writing to cache (first call) | $1.25 / million (1.25× — small surcharge) |
| **Reading from cache (later calls)** | **$0.10 / million (0.1× — 90% off!)** |

Anthropic's own launch numbers: up to **90% cost reduction** and up to **85% latency reduction** on long prompts. For Priya, the 2,000-token menu+policy block that every customer request re-sends now costs one-tenth as much from the second call onward — *and* responses start faster.

### How does it work?

1. You mark a **cache breakpoint** in your request with `cache_control: {"type": "ephemeral"}`.
2. First call: everything up to the breakpoint is processed and **written** to the cache (1.25× price).
3. Later calls with the **exact same prefix**: it's **read** from cache (0.1× price). Each read also refreshes the timer, free.
4. The cache lives **5 minutes** by default (a 1-hour option costs 2× to write). After that it expires — remember, this is a *cost* optimization, not a memory system!

### The four rules that make or break caching

1. **Exact match only.** One changed character in the cached part = cache miss. Put *stable* content (system prompt, tools, policies) first, *changing* content (the new question) last.
2. **Order is fixed:** tools → system → messages. A change higher up invalidates everything below it.
3. **Minimum size.** Below a per-model minimum, nothing is cached at all (no error — it just silently doesn't cache). **On Haiku 4.5 the minimum is 4,096 tokens.** Our lab builds a big enough prompt on purpose.
4. **Put the breakpoint on content that doesn't change.** Mark the end of the *stable* prefix — not the user's new message, which changes every call and can never match.

### How do I know it worked? Read the receipt

Every response includes `usage` fields — your cache receipt:

- `cache_creation_input_tokens` — tokens **written** to cache this call
- `cache_read_input_tokens` — tokens **read** from cache this call (the 90%-off part)
- `input_tokens` — tokens after the breakpoint, billed normally

### 💡 Remember this

> *"Caching doesn't make Claude remember — it makes re-sending what you were already re-sending 90% cheaper and much faster. Memory is your job; caching is your discount."*

### ❌ Don't mix these up

- ❌ "Prompt caching gives Claude long-term memory." → ✅ It's a cost/latency optimization on content **you still send every time**. Memory stays your app's job.
- ❌ "Cached responses are lower quality." → ✅ Output is **identical** with or without caching — only price and speed change.
- ❌ "I cached my 500-token system prompt on Haiku." → ✅ Below the 4,096-token minimum on Haiku 4.5, nothing is cached (silently). Check the usage fields.

### 🤯 Fun fact

Anthropic launched prompt caching on **August 14, 2024**. One highlighted use case: caching an *entire book*. Anthropic's example showed that chatting with the full text of *Pride and Prejudice* cached went from ~12 seconds of latency to ~2.4 seconds — an 80% speedup, plus 90% off the input bill.


## 💻 Lab 5 — See the 90% discount in the usage receipt

**Objective:** build a TastyTiffin policy pack big enough to cache (>4,096 tokens on Haiku 4.5), mark it with a cache breakpoint, and watch the usage numbers flip from *write* to *read*.


In [9]:
# Lab 5a: build a policy pack big enough to cross Haiku 4.5's 4,096-token cache minimum

policy_chunk = (
    "TastyTiffin Refund Policy: cold food qualifies for a full refund or wallet credit "
    "within 24 hours of delivery. Late delivery beyond 45 minutes earns a 20% coupon. "
    "Menu: sambar rice, curd rice, chapati with kurma, lemon rice, veg biryani, "
    "millet pongal, rasam rice, and Friday special: paneer butter masala with parotta. "
    "Delivery windows: lunch 11:30-14:00, dinner 18:30-21:30, across Chennai. "
)

big_policy = "You are Mitra, TastyTiffin's support assistant. Follow these policies.\n\n"
big_policy += policy_chunk * 60    # repeat to comfortably exceed the 4,096-token minimum

count = client.messages.count_tokens(
    model=MODEL,
    system=big_policy,
    messages=[{"role": "user", "content": "hi"}],
)
print("Prompt size:", count.input_tokens, "tokens (need > 4096 to cache on Haiku 4.5)")

Prompt size: 7410 tokens (need > 4096 to cache on Haiku 4.5)


In [10]:
# Lab 5b: call twice — first call WRITES the cache, second call READS it

def ask_mitra(question):
    reply = client.messages.create(
        model=MODEL,
        max_tokens=100,
        system=[{
            "type": "text",
            "text": big_policy,
            "cache_control": {"type": "ephemeral"},   # breakpoint: cache up to HERE
        }],
        messages=[{"role": "user", "content": question}],
    )
    u = reply.usage
    print("  cache WRITE tokens :", u.cache_creation_input_tokens)
    print("  cache READ  tokens :", u.cache_read_input_tokens)
    print("  normal input tokens:", u.input_tokens)
    print("  Mitra:", reply.content[0].text[:120], "\n")

print("--- Call 1 (expect a big WRITE, zero READ) ---")
ask_mitra("My order #4521 arrived cold. What are my options?")

print("--- Call 2 (expect zero WRITE, big READ = 90% off) ---")
ask_mitra("Is veg biryani on the menu?")

--- Call 1 (expect a big WRITE, zero READ) ---
  cache WRITE tokens : 7403
  cache READ  tokens : 0
  normal input tokens: 20
  Mitra: I'm sorry to hear your order arrived cold! 🙁

According to our refund policy, you have two options:

1. **Full Refund**  

--- Call 2 (expect zero WRITE, big READ = 90% off) ---
  cache WRITE tokens : 0
  cache READ  tokens : 7403
  normal input tokens: 16
  Mitra: Yes, veg biryani is on our menu! 🍚

You can order it during our delivery windows:
- **Lunch**: 11:30 AM - 2:00 PM
- **Di 



**Expected result:**

- **Call 1:** `cache_creation_input_tokens` ≈ several thousand (written at 1.25×), `cache_read_input_tokens` = 0.
- **Call 2:** `cache_creation_input_tokens` = 0, `cache_read_input_tokens` ≈ the same several thousand — now billed at **$0.10/MTok instead of $1.00**. The tiny `input_tokens` is just the new question.

**Try this 🔧:** (1) Wait 6+ minutes and call again — the cache expired, so you'll see a fresh WRITE. (2) Change one word inside `big_policy` and call — cache miss, fresh WRITE. Exact match matters.

> **Also good to know:** the docs' newer *automatic caching* option puts `cache_control={"type": "ephemeral"}` at the **top level** of the request, and the API auto-places the breakpoint on the last cacheable block — ideal for growing multi-turn conversations. We used an **explicit** breakpoint here because our stable part (the policy) is followed by a question that changes every call, and the breakpoint must sit on content that *doesn't* change. Knowing when to use which is an architect-level detail.

### ✅ Checkpoint

> **Q:** Priya's nightly batch job runs once per day with the same giant system prompt. Will the 5-minute cache help across runs on different days?

<details><summary>Show answer</summary>

**No.** The cache lives 5 minutes (or 1 hour with the paid `ttl: "1h"` option). Runs a day apart always re-write the cache. Caching helps *bursts* of similar requests, not widely spaced ones.
</details>

### 🎤 Interview angle

**Q (FDE): "A client says: 'Caching sounds like it stores our customer data on your servers — compliance won't allow it.'"**
*Model answer:* "The cache holds the model's processed state for a prefix, in memory, for minutes — with cryptographic hashes for matching, isolated per organization (and per workspace), and it's even eligible under Anthropic's zero-data-retention terms. I'd frame it as a short-lived performance layer, not data storage, and show the compliance team the docs' data-retention section."


# Section 7 — How Real Products Do It (and the claude.ai Bridge)

Everything you built today runs inside the biggest AI products, right now:

| What you built today | Where you've already seen it |
|---|---|
| Conversation list replayed each turn | Every chat on **claude.ai** — the app resends your thread to the API each message |
| Facts pinned so they survive trimming | **claude.ai Projects** (project knowledge sent with every chat) and memory features |
| Sliding window / trimming | Long claude.ai chats eventually hit a length limit — same context window you measured |
| Prompt caching | claude.ai, Claude Code, and most serious API products cache system prompts and tools under the hood |

**Beyond today (know these exist — one line each):**

- **Summarization memory:** ask the model to summarize dropped turns and keep the summary in context — recall without the token bill.
- **Anthropic's memory tool + context editing:** newer API features where Claude itself reads/writes memory files across conversations and old tool results are auto-cleared. Built from the same primitives you just learned.
- **RAG (a later session):** store knowledge in a database and retrieve only the relevant bits per request — memory that scales beyond any context window.

> If you understand today's list + window + cache, none of those will ever feel like magic — they're the same three ideas, industrialized.


# Architecture — Mitra in Production

```
 Customer (app / WhatsApp)
        |
        v
 +--------------------+     load/save history      +------------------+
 |  Priya's backend   | <------------------------> |  Database        |
 |  (the "state" home)|      per customer          |  (chat history,  |
 |                    |                            |   facts/profile) |
 |  1. load history   |                            +------------------+
 |  2. sliding window |
 |  3. build request: |
 |     [cached: system+policies] + [history] + [new msg]
 |  4. call Claude    |
 +---------|----------+
           v
   Claude API (STATELESS)
   - prompt cache (5 min): policies read at 0.1x price
   - 200K context window (Haiku 4.5)
```

**Component responsibilities:** the backend owns *state* (load, trim, save); the database owns *durability* (history survives restarts); the Claude API owns *intelligence* (and holds zero conversation state).

**Failure points an architect names in review:**
- Database down → bot still answers but with amnesia. Decide: fail closed or degrade gracefully?
- Window trims a critical fact → wrong answers. Mitigate: pin facts to the system prompt / profile store.
- Someone "just edits" the system prompt at noon → every cache misses at once → cost and latency spike. Mitigate: version prompts, deploy in low-traffic windows.
- Trimming bug breaks user-first/alternating order → hard API errors. Mitigate: pair-safe trimming + a validation check before send.

**Cost levers, in the order an architect pulls them:** smaller model (Haiku) → prompt caching → tighter window → summarization → shorter system prompt.


# 🧪 Main Claude API Lab — Mitra v3: Memory + Window + Cache Together

**Objective:** combine all three techniques in one production-shaped bot, and print the cache receipt every turn.


In [11]:
# Mitra v3 — the full session in ~35 lines

class MitraBotV3:
    def __init__(self, system_text, max_messages=8):
        self.system_blocks = [{
            "type": "text",
            "text": system_text,
            "cache_control": {"type": "ephemeral"},   # cache the stable prefix
        }]
        self.history = []
        self.max_messages = max_messages

    def chat(self, user_text):
        self.history = sliding_window(self.history, self.max_messages)  # from Lab 4
        self.history.append({"role": "user", "content": user_text})
        reply = client.messages.create(
            model=MODEL,
            max_tokens=200,
            system=self.system_blocks,
            messages=self.history,
        )
        answer = reply.content[0].text
        self.history.append({"role": "assistant", "content": answer})
        u = reply.usage
        print(f"  [receipt] write={u.cache_creation_input_tokens} "
              f"read={u.cache_read_input_tokens} normal={u.input_tokens}")
        return answer

mitra3 = MitraBotV3(big_policy)   # the >4096-token policy pack from Lab 5

for q in [
    "Hi, I'm Karthik. Order #4521 arrived cold.",
    "What refund options do I have?",
    "I'll take the wallet credit. Also, is millet pongal available at dinner?",
    "Great. Summarize everything we agreed today.",
]:
    print("Karthik:", q)
    print("Mitra :", mitra3.chat(q), "\n")

Karthik: Hi, I'm Karthik. Order #4521 arrived cold.
  [receipt] write=0 read=7403 normal=23
Mitra : Hi Karthik! I'm sorry to hear your order arrived cold. That's not the experience we want you to have.

Good news—according to our refund policy, **cold food qualifies for a full refund or wallet credit within 24 hours of delivery**. 

Here's what I can do for you:

1. **Full Refund** - Return the money to your original payment method
2. **Wallet Credit** - Add the full amount to your TastyTiffin wallet for future orders

Which option would you prefer? Once you let me know, I'll process this right away for you. 

Karthik: What refund options do I have?
  [receipt] write=0 read=7403 normal=173
Mitra : Great question, Karthik! For your cold food order, you have **two refund options**:

1. **Full Refund** - We'll return the complete order amount to your original payment method
2. **Wallet Credit** - We'll add the full order amount as credit to your TastyTiffin wallet, which you can use for a

**Expected result:** turn 1 shows a cache **write**; turns 2–4 show cache **reads** (90% off the policy pack) while memory works across all turns — the final summary should mention the cold order, #4521, and the wallet credit.

**Try this 🔧:** drop `max_messages` to 2 and rerun. Watch the receipt stay cheap but the final summary lose early details. You are now *tuning* the memory/cost tradeoff — that's the actual day job.

### 💰 A one-line cost note

This whole lab — a dozen Haiku calls with a ~5K-token cached prompt — costs about a cent. Caching is why.


# 🚀 Mini Project — TastyTiffin Support Copilot

**Business use case:** Priya wants one bot that handles a full support shift: greets customers, remembers each customer separately, survives 30+ turn conversations, and keeps API spend flat as traffic grows.

**Architecture (build exactly this):**

```
customers (many) --> SupportDesk
                      - bots: dict  {customer_id -> MitraBotV3}
                      - one shared cached policy prompt (write once, read cheap)
                      - per-customer history + pair-safe window
```

**Build steps:**
1. Wrap `MitraBotV3` in a `SupportDesk` class holding a `dict` of bots keyed by customer ID — memory isolation per customer (Karthik's refund must never leak into Divya's chat).
2. Simulate two interleaved customers (alternate their messages) and prove isolation: ask each "what did I complain about?"
3. Add fact-pinning: after each turn, if the message contains an order number, append it to that customer's system block — so it survives the window. (Careful: what does editing the system text do to the cache? You know the answer now.)
4. Print a per-conversation cost report from the usage receipts: total write, read, and normal tokens → estimated ₹/$ cost.

**Definition of done:** two customers chat 6+ turns interleaved with correct isolated memory; cache reads appear from each customer's second turn; the cost report shows cached turns ≈ 90% cheaper on the policy portion; a 3-sentence written answer to: "what breaks first at 100× traffic?"


# 📋 Session Summary

Claude's API is **stateless**: every request stands alone, so "memory" must be built by your application. The universal pattern is the **conversation list** — resend every user and assistant message each turn. That list grows, colliding with the **context window** (200K tokens on Haiku 4.5) and with **per-token billing on every call**. The **sliding window** caps growth by dropping the oldest user+assistant *pairs* (never break the user-first, alternating rule) at the price of forgetting old facts — so durable facts get pinned in the system prompt or a profile store. **Prompt caching** attacks the other half of the bill: mark the stable prefix with `cache_control` and re-reads cost 0.1× (90% off) with much lower latency, for 5 minutes per refresh (1 hour paid), minimum 4,096 tokens on Haiku 4.5. Memory is your job; caching is your discount; the window is your budget.


# ✅ What You Learned Today

You can now:

- [ ] Explain stateless vs stateful — and *whose job* conversation memory is
- [ ] Prove statelessness with two API calls
- [ ] Implement memory with a `messages` list and the role/content format
- [ ] Build a chatbot class (Mitra) with a system prompt and growing history
- [ ] Count tokens exactly with `client.messages.count_tokens(...)`
- [ ] Explain why cost grows every turn with list-based memory
- [ ] Implement a pair-safe sliding window and name its failure mode
- [ ] Enable prompt caching with `cache_control` and read the usage receipt
- [ ] State the caching numbers: 1.25× write, 0.1× read, 5-min TTL, 4,096-token minimum on Haiku 4.5
- [ ] Sketch the production architecture: backend owns state, DB owns durability, API owns intelligence


# 🗂️ AI Architect Cheat Sheet

**Definitions**

| Term | One-liner |
|---|---|
| Stateless | Server keeps nothing between requests; each request self-contained |
| Stateful | Information carried across requests (your app's job) |
| Context window | Max tokens per request — 200K (Haiku 4.5); 1M on Sonnet 4.6/5 & Opus 4.8 |
| Sliding window | Keep last N messages, drop oldest pairs |
| Prompt caching | Server reuses processed stable prefix; 5-min default TTL |

**Key numbers (Haiku 4.5, verified July 2026)**

| Item | Value |
|---|---|
| Input / output price | $1 / $5 per MTok |
| Cache write / read | $1.25 (1.25×) / $0.10 (0.1×) per MTok |
| Cache TTL | 5 min default; 1 hour at 2× write price |
| Min cacheable prompt | 4,096 tokens (Haiku 4.5) — silently skipped below |
| Max cache breakpoints | 4 |
| Token rule of thumb | 1 token ≈ 3.5 English chars ≈ 3/4 word |

**Decision table**

| Symptom | Reach for |
|---|---|
| Bot forgets between turns | Conversation list (send history) |
| Hitting context limit / cost creeping per turn | Sliding window (+ summarize or pin facts) |
| Same big prefix sent repeatedly | Prompt caching (breakpoint on stable content) |
| Facts must survive forever | Pin to system prompt / DB profile (later: memory tool, RAG) |

**API quick-reference**

```python
client.messages.create(model=..., max_tokens=..., system=..., messages=[...])
client.messages.count_tokens(model=..., messages=[...])          # free, exact
system=[{"type":"text","text":BIG,"cache_control":{"type":"ephemeral"}}]
reply.usage.cache_creation_input_tokens / cache_read_input_tokens / input_tokens
```


# ⏱️ 5-Minute Revision Guide

1. **Claude forgets by design.** The API is stateless — for privacy, scale, reproducibility. Memory is the app's job.
2. **The fix is a list.** Store every `{"role","content"}` message; resend all of it each call. User first, roles alternate.
3. **The list bites back.** You re-pay input tokens for the whole history every turn; the context window (200K on Haiku 4.5) is the ceiling.
4. **Sliding window = trim in pairs.** Caps cost; forgets old facts — pin important ones to the system prompt.
5. **Caching = 90% off the stable prefix.** `cache_control: ephemeral`; write 1.25×, read 0.1×; 5-min TTL; exact-match only; ≥4,096 tokens on Haiku 4.5; verify via `usage`.
6. **The one sentence:** *memory is your job, caching is your discount, the window is your budget.*


# 🎤 Interview Preparation Notes

**Q1. Is the Claude API stateful or stateless, and why does it matter?**
Stateless — each request is independent and must contain the full conversation. It matters because memory, cost control, and context management all become application-layer responsibilities.

**Q2. How do you implement multi-turn memory?**
Maintain an ordered list of user/assistant messages, append both sides every turn, resend the entire list. Persist it in a database keyed by user/session for durability.

**Q3. Conversation exceeds the context window — what are your options and tradeoffs?**
Sliding window (cheap, forgets old facts), summarization (recall at small token cost, adds a model call), fact-pinning to system prompt/profile (durable, needs extraction logic), and eventually RAG. Production systems combine them.

**Q4. Explain prompt caching pricing and when it pays off.**
Write costs 1.25× base input, reads cost 0.1×. It pays off from the very first re-read within the TTL (5 min default, 1 h at 2×). Best for stable prefixes reused in bursts: system prompts, tools, policies, long documents, growing chat history.

**Q5. Why can a cache silently not work?**
Below the model's minimum (4,096 tokens on Haiku 4.5), prefix not byte-identical, breakpoint placed on content that changes each request, or TTL expired. Diagnose with the `usage` fields: both cache numbers 0 → nothing cached.

**Q6 (architecture). Design memory for a support bot at 1M conversations/month.**
Backend loads history from a DB per conversation, applies pair-safe windowing + summary, pins durable facts to a profile, marks the shared system/policy prefix with a cache breakpoint, monitors usage receipts for cache hit-rate, and versions prompt changes to avoid cache stampedes.

**Q7 (FDE). Client: "the bot forgot what the user said 40 turns ago — your product is broken."**
Reframe: the model reads only what we send; we tuned the window for cost. Options with price tags: widen the window, add summarization, pin key facts. Demo the fix live with the usage receipt showing the cost impact of each.

**Q8 (FDE). Client worries caching stores their data.**
Cache = short-lived in-memory processed state + hashes, isolated per org/workspace, minutes-long TTL, ZDR-eligible. Show the docs' data-retention section; position as performance layer, not storage.


# 📝 Assignment

**Beginner —** Rebuild `MitraBot` from scratch without looking, for a different business (a gym's front-desk bot). Prove memory with a 3-turn chat.

**Intermediate —** Add `count_tokens` to `MitraBotV2` so it prints per-turn input size, and trigger the sliding window by *token count* (e.g., > 1,500 tokens) instead of message count. Keep it pair-safe.

**Advanced —** Implement **summarization memory**: when the window trims messages, send the dropped turns to Haiku with "summarize in 2 sentences", and keep the running summary as the first user message. Show the bot answering a question about a trimmed turn.

**Project —** Complete the TastyTiffin Support Copilot (mini project spec above), including the per-customer cost report and the "what breaks at 100× traffic" write-up.


# 🧪 Assessment

### Part A — Multiple choice (10)

**1.** "The Claude API is stateless" means:
(a) It can't follow instructions (b) Each request is independent; no conversation is stored between calls (c) It has no system prompt (d) It forgets mid-response

**2.** To make a chatbot remember, your app must:
(a) Enable `memory=True` (b) Use a bigger model (c) Resend the full message history each call (d) Call the same server each time

**3.** A valid `messages` list must:
(a) Start with an assistant message (b) Start with a user message, roles alternating (c) Contain only user messages (d) Be under 10 messages

**4.** Claude Haiku 4.5's context window is:
(a) 4,096 tokens (b) 32K (c) 200K (d) Unlimited

**5.** With list-based memory, the input cost of turn 20:
(a) Equals turn 1's cost (b) Includes tokens from all previous turns (c) Is free after caching (d) Only counts output tokens

**6.** A sliding window should trim:
(a) Newest messages (b) Random messages (c) Oldest user+assistant pairs (d) Only assistant messages

**7.** Prompt caching's read price is:
(a) Free (b) 0.1× base input (c) 1.25× base input (d) 2× base input

**8.** The default cache lifetime is:
(a) 5 minutes (b) 1 hour always (c) 24 hours (d) Forever

**9.** On Haiku 4.5, a 500-token system prompt marked with `cache_control`:
(a) Caches normally (b) Throws an error (c) Is silently not cached — below the 4,096-token minimum (d) Caches at 2× price

**10.** `cache_read_input_tokens: 5200, cache_creation_input_tokens: 0` means:
(a) Cache miss (b) Cache expired (c) 5,200 tokens were reused from cache at 90% off (d) 5,200 tokens were written

### Part B — Short answer (5)

**11.** Why does claude.ai "remember" your conversation when the API is stateless?
**12.** Name the two separate problems caused by an ever-growing history list.
**13.** Why must a sliding window trim in pairs?
**14.** Why should the cache breakpoint never sit on the user's newest message?
**15.** Caching vs memory: one sentence on the difference.

### Part C — Scenarios (3)

**16.** Priya edits one sentence of the cached policy prompt at 12:05 pm during lunch rush. Predict the next 5 minutes of cost and latency, and propose a safer deployment approach.

**17.** A customer chats for 60 turns. Around turn 40, Mitra starts contradicting things agreed in turns 1–10, and by turn 55 the API rejects a request outright. Diagnose both symptoms and propose a combined fix.

**18.** Finance says the bot's API bill doubled after "someone improved the system prompt". The prompt grew from 3,900 to 6,000 tokens — on Sonnet 4.6 calls it got *cheaper per call*, but the Haiku 4.5 fleet got more expensive. Wait — explain why growing a prompt could ever *reduce* cost, and what the Haiku fleet's problem might be. *(Hint: minimum cacheable sizes: Sonnet 4.6 = 1,024; Haiku 4.5 = 4,096.)*


# 🔑 Answer Key

**Part A:** 1-b · 2-c · 3-b · 4-c · 5-b · 6-c · 7-b · 8-a · 9-c · 10-c

**Part B:**
**11.** The claude.ai *application* stores your thread and resends it to the API with every message — app-layer statefulness over a stateless API.
**12.** (1) Rising cost: full history is re-billed as input every call. (2) A hard ceiling: the context window eventually rejects or truncates the request.
**13.** The API expects user-first, alternating roles; trimming an odd number can leave the list starting with an assistant message → errors.
**14.** The newest message changes every request, so the prefix hash never matches → you pay cache *writes* every call and never get a read. Breakpoint belongs at the end of the *stable* content.
**15.** Memory decides *what the model sees* (your app's job); caching makes *re-sending it cheaper and faster* (Anthropic's discount) — caching stores nothing for you long-term.

**Part C:**
**16.** Every request's cached prefix now mismatches → all traffic pays fresh 1.25× cache writes with full processing latency until the new prefix is warm; during rush this is a visible cost + latency spike. Safer: version prompts, deploy during a low-traffic window, or pre-warm the cache before switching traffic.
**17.** Turn-40 contradictions: the sliding window (or truncation) dropped early turns, so agreements from turns 1–10 vanished. Turn-55 rejection: history finally exceeded the context window (or broke message-order rules after a bad trim). Combined fix: pair-safe window + running summary of dropped turns + pin durable facts (order #, promises) to the system prompt/profile.
**18.** At 3,900 tokens the prompt was *below* Haiku 4.5's 4,096-token minimum — it was never cached, every call paid full price. Growing it past 4,096 would actually *enable* caching (as it already did on Sonnet 4.6, whose minimum is 1,024). If the Haiku fleet still got pricier, likely causes: the breakpoint sits on changing content, calls are >5 min apart (TTL expiry), or the prompt isn't byte-identical across servers. The lesson: a *bigger* prompt that caches can be cheaper than a *smaller* one that doesn't — check the usage receipts, not the prompt length.
